In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("diabetic_clean.csv", dtype={"diag_1": str})
print("Shape:", df.shape)

Shape: (69990, 50)


In [3]:
for col in ["A1Cresult", "max_glu_serum"]:
    df[col] = df[col].fillna("None")

In [4]:
def diag_group(code):
    if pd.isna(code):
        return "Unknown"
    code = str(code)
    if code.startswith(("V", "E")):
        return "Other"
    if code.startswith("250"):
        return "Diabetes"
    n = int(float(code))
    if 390 <= n <= 459 or n == 785:
        return "Circulatory"
    if 460 <= n <= 519 or n == 786:
        return "Respiratory"
    if 520 <= n <= 579 or n == 787:
        return "Digestive"
    if 580 <= n <= 629 or n == 788:
        return "Genitourinary"
    if 710 <= n <= 739:
        return "Musculoskeletal"
    if 800 <= n <= 999:
        return "Injury"
    if 140 <= n <= 239:
        return "Neoplasms"
    return "Other"
 
df["diag_group"] = df["diag_1"].apply(diag_group)

In [5]:
df["age_lo"] = df["age"].str.extract(r"\[(\d+)-")[0].astype(int)
df["age_group"] = pd.cut(
    df["age_lo"], bins=[-1, 39, 59, 69, 79, 200],
    labels=["<40", "40-59", "60-69", "70-79", "80+"],
).astype(str)

In [6]:
df["prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
df["prior_inpatient_band"] = pd.cut(
    df["number_inpatient"], bins=[-1, 0, 1, 2, 1000], labels=["0", "1", "2", "3+"]
).astype(str)
df["los_band"] = pd.cut(
    df["time_in_hospital"], bins=[0, 3, 7, 1000], labels=["1-3 days", "4-7 days", "8+ days"]
).astype(str)
df["med_band"] = pd.cut(
    df["num_medications"], bins=[-1, 10, 20, 1000], labels=["0-10", "11-20", "21+"]
).astype(str)

In [7]:
df["a1c_status"] = df["A1Cresult"].replace({"None": "Not tested", "Norm": "Normal"})
df["a1c_tested"] = (df["A1Cresult"] != "None").astype(int)
df["med_change"] = df["change"].map({"Ch": "Changed", "No": "No change"})

In [8]:
df["discharge_group"] = (
    df["discharge_disposition_id"]
    .map({1: "Home", 6: "Home + home health", 3: "Nursing facility (SNF)"})
    .fillna("Other")
)

In [9]:
for col in ["diag_group", "age_group", "prior_inpatient_band", "los_band",
            "med_band", "a1c_status", "med_change", "discharge_group"]:
    print("\n" + col)
    print(df[col].value_counts(dropna=False))


diag_group
diag_group
Circulatory        21390
Other              12124
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4696
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Unknown               10
Name: count, dtype: int64

age_group
age_group
40-59    19179
70-79    17751
60-69    15689
80+      12871
<40       4500
Name: count, dtype: int64

prior_inpatient_band
prior_inpatient_band
0     61795
1      5798
2      1501
3+      896
Name: count, dtype: int64

los_band
los_band
1-3 days    35146
4-7 days    25075
8+ days      9769
Name: count, dtype: int64

med_band
med_band
11-20    35046
0-10     19499
21+      15445
Name: count, dtype: int64

a1c_status
a1c_status
Not tested    57144
>8             6239
Normal         3741
>7             2866
Name: count, dtype: int64

med_change
med_change
No change    38493
Changed      31497
Name: count, dtype: int64

discharge_group
discharge_group
Home                    

In [10]:
df.to_csv("diabetic_features.csv", index=False)
print("Saved diabetic_features.csv")

Saved diabetic_features.csv
